In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [1]:
url = 'https://raw.githubusercontent.com/ichiP245/my-next-soderling/refs/heads/main/Archivos/df_tenis_columns_OK.csv'

In [3]:
df = pd.read_csv(url)

In [61]:
df['Fecha'] = pd.to_datetime(df['Fecha'], format='%Y-%m-%d')

## Creacion de Variables

Implied Probabilities de cada campo que tiene cuotas (odds)

In [23]:
def implied_prob_adjusted(odds_A, odds_B):
    """Pasamos odds decimales en probabilidades ajustadas eliminando el margen del bookmaker."""
    inv_A = 1 / odds_A
    inv_B = 1 / odds_B

    total = inv_A + inv_B

    prob_A = inv_A / total
    prob_B = inv_B / total

    return prob_A, prob_B

In [29]:
probA = 1 / df['B365A']
probB = 1 / df['B365B']
df['B365BookmakersMargin'] = (probA + probB - 1)

df['B365ProbA'], df['B365ProbB'] = implied_prob_adjusted(df['B365A'], df['B365B'])
df['ProbAvgA'], df['ProbAvgB'] = implied_prob_adjusted(df['AvgA'], df['AvgB'])
df['ProbMaxA'], df['ProbMaxB'] = implied_prob_adjusted(df['MaxA'], df['MaxB'])

df['market_uncertainty'] = np.abs(df['ProbAvgA'] - 0.5) # Mas bajo, mas parejo el partido

# Vemos la diferencia entre las probabilidades mas optimistas y las del consenso general del mercado
df['SpreadOddsA'] = df['ProbMaxA'] - df['ProbAvgA']
df['SpreadOddsB'] = df['ProbAvgB'] - df['ProbMaxB']

# Diferencia promedio de probabilidades
df['AvgOddsDiff'] = (df['ProbAvgA'] - df['ProbAvgB'])

# Probabilidades con un logaritmo aplicado, recomendado por ChatGPT
df['logit_oddsA'] = np.log(df['ProbAvgA'] / (1 - df['ProbAvgA']))
df['logit_oddsB'] = np.log(df['ProbAvgB'] / (1 - df['ProbAvgB']))

Ranking Difference, ATP Points Difference

In [30]:
df['rankDiff'] = df['rankA'] - df['rankB']
df['ptsDiff'] = df['PtsA'] - df['PtsB']

Porcentaje de partidos ganados en los últimos 5, 10 o 20 partidos

In [65]:
jug = pd.concat([df['playerA'], df['playerB']], axis=0).value_counts()
print(f'Total: {jug.shape[0]} jugadores')
print(f'Con mas de 20 partidos: {jug[jug>20].shape[0]} jugadores')
print(f'Con mas de 10 partidos: {jug[jug>10].shape[0]} jugadores')
print(f'Con mas de 5 partidos: {jug[jug>5].shape[0]} jugadores')

Total: 613 jugadores
Con mas de 20 partidos: 264 jugadores
Con mas de 10 partidos: 325 jugadores
Con mas de 5 partidos: 385 jugadores


In [64]:
a = df[['Fecha','playerA','target']].rename(columns={'playerA':'Player','target':'win'})
b = df[['Fecha','playerB','target']].rename(columns={'playerB':'Player','target':'win'})
b['win'] = 1 - b['win']

In [66]:
matches_long = pd.concat([a[["Fecha", "Player", "win"]],
                          b[["Fecha", "Player", "win"]]],
                         ignore_index=True).sort_values(["Player", "Fecha"])

matches_long["matches_played_before"] = (matches_long.groupby("Player").cumcount())

matches_long["wins_before"] = (matches_long.groupby("Player")["win"].cumsum().shift(1))

matches_long["wins_before"] = matches_long["wins_before"].fillna(0)

matches_long["win_pct_before"] = (matches_long["wins_before"] / matches_long["matches_played_before"].replace(0, pd.NA))

In [ ]:
def calculate_rolling_win_pct(df_long, window_size):
    # Sort by player and date to ensure correct rolling calculation
    df_sorted = df_long.sort_values(by=['Player', 'Fecha']).copy()

    # Calculate rolling sum of wins for the given window size
    df_sorted[f'rolling_wins_{window_size}'] = df_sorted.groupby('Player')['win'].transform(lambda x: x.rolling(window=window_size, closed='left').sum())

    # Calculate rolling count of matches for the given window size
    df_sorted[f'rolling_matches_{window_size}'] = df_sorted.groupby('Player')['win'].transform(lambda x: x.rolling(window=window_size, closed='left').count())

    # Calculate rolling win percentage, handling division by zero
    df_sorted[f'rolling_win_pct_{window_size}'] = df_sorted[f'rolling_wins_{window_size}'] / df_sorted[f'rolling_matches_{window_size}']

    # Fill NaN values (for players with fewer than 'window_size' matches) with 0 or a suitable default
    df_sorted[f'rolling_win_pct_{window_size}'] = df_sorted[f'rolling_win_pct_{window_size}'].fillna(0)
    df_sorted[f'rolling_wins_{window_size}'] = df_sorted[f'rolling_wins_{window_size}'].fillna(0)
    df_sorted[f'rolling_matches_{window_size}'] = df_sorted[f'rolling_matches_{window_size}'].fillna(0)

    return df_sorted[['Fecha', 'Player', f'rolling_wins_{window_size}', f'rolling_matches_{window_size}', f'rolling_win_pct_{window_size}']]

# Calculate rolling win percentages for different window sizes
rolling_win_pct_5 = calculate_rolling_win_pct(matches_long, 5)
rolling_win_pct_10 = calculate_rolling_win_pct(matches_long, 10)
rolling_win_pct_20 = calculate_rolling_win_pct(matches_long, 20)

# Merge these back into the original matches_long DataFrame if needed, or process separately
# For this task, I will merge them back to see the results clearly.
matches_long = pd.merge(matches_long, rolling_win_pct_5[['Fecha', 'Player', 'rolling_win_pct_5']], on=['Fecha', 'Player'], how='left')
matches_long = pd.merge(matches_long, rolling_win_pct_10[['Fecha', 'Player', 'rolling_win_pct_10']], on=['Fecha', 'Player'], how='left')
matches_long = pd.merge(matches_long, rolling_win_pct_20[['Fecha', 'Player', 'rolling_win_pct_20']], on=['Fecha', 'Player'], how='left')

In [99]:
matches_long.shape[0]

35146

In [100]:
df.shape[0]*2

32122

Sets jugados en ultimos 30 dias

Sets jugados en ultimos 5 dias

Partidos en ultimos 365 dias

Dias desde el ultimo partido

Diferencia de dias desde el ultimo partido (Jugador A - Jugador B)

Racha actual de victorias

Record en cada superficie, previo al partido

Cantidad de partidos previos entre ambos

## Encoding de categoricas

is_grand_slam
is_indoor
is_clay
is_best_of_5